# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [ ]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5010 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [ ]:
import functools

def count_elements(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args:
            if isinstance(arg, list):
                print(f"Liczba elementów: {len(arg)}")
        
        for value in kwargs.values():
            if isinstance(value, list):
                print(f"Liczba elementów: {len(value)}")
                
        return func(*args, **kwargs)
    return wrapper

@count_elements
def moja_funkcja(dane):
    return dane

moja_funkcja([1, 2, 3])

Liczba elementów: 3


[1, 2, 3]

### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [ ]:
import functools
import datetime
import time

def log_to_file(nazwa_pliku):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start = time.time()
            wynik = func(*args, **kwargs)
            koniec = time.time()
            
            czas_trwania = koniec - start
            data = datetime.datetime.now()
            
            with open(nazwa_pliku, "a") as f:
                f.write(f"{func.__name__}; {data}; {czas_trwania}\n")
            
            return wynik
        return wrapper
    return decorator

@log_to_file("moje_logi.log")
def przyklad():
    time.sleep(0.5)
    print("Funkcja działa")

przyklad()

Funkcja dziła


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [ ]:
class EmailValidator:
    def __set__(self, instance, value):
        if "@" not in value or "." not in value:
            raise ValueError("Niepoprawny format email!")
        instance.__dict__['email'] = value

    def __get__(self, instance, owner):
        return instance.__dict__.get('email')

class Student:
    email = EmailValidator()

    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

s = Student("Jan", "Kowalski", "jan@pwr.edu.pl")
print(s.email)


jan@pwr.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [9]:
class AccessLogger:
    def __init__(self, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        wartosc = instance.__dict__.get(self.name)
        print(f"ODCZYT: Atrybut '{self.name}' ma wartość: {wartosc}")
        return wartosc

    def __set__(self, instance, value):
        print(f"ZAPIS: Zmieniam '{self.name}' na: {value}")
        instance.__dict__[self.name] = value

class Uzytkownik:
    imie = AccessLogger("imie")
    wiek = AccessLogger("wiek")

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

# Test
u = Uzytkownik("Marek", 21)
print(u.imie)
u.wiek = 22

ZAPIS: Zmieniam 'imie' na: Marek
ZAPIS: Zmieniam 'wiek' na: 21
ODCZYT: Atrybut 'imie' ma wartość: Marek
Marek
ZAPIS: Zmieniam 'wiek' na: 22


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [10]:
class Fibonacci:
    def __init__(self, limit):
        self.limit = limit
        self.count = 0
        self.a, self.b = 0, 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        wartosc = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return wartosc

# Test
fib = Fibonacci(10)
for liczba in fib:
    print(liczba)

0
1
1
2
3
5
8
13
21
34


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [13]:
class Collatz:
    def __init__(self, n):
        self.n = n
        self.koniec = False

    def __iter__(self):
        return self

    def __next__(self):
        if self.koniec:
            raise StopIteration
            
        aktualna = self.n
        
        if self.n == 1:
            self.koniec = True
        elif self.n % 2 == 0:
            self.n = self.n // 2
        else:
            self.n = 3 * self.n + 1
            
        return aktualna

for liczba in Collatz(12):
    print(liczba)

12
6
3
10
5
16
8
4
2
1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [15]:
import functools

current_user = {"username": "student", "role": "user"}

def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user.get("role") != role:
                raise PermissionError(f"Brak uprawnień! Wymagana rola: {role}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@require_role("admin")
def usun_baze():
    return "Baza usunięta"

# Test
try:
    print(usun_baze())
except PermissionError as e:
    print(e)

Brak uprawnień! Wymagana rola: admin


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [30]:
class Typed:
    def __init__(self, typ):
        self.typ = typ
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if not isinstance(value, self.typ):
            raise TypeError(f"Atrybut {self.name} musi być typu {self.typ.__name__}")
        instance.__dict__[self.name] = value

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

class Osoba:
    wiek = Typed(int)
    imie = Typed(str)

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

# Test
o = Osoba("Marek", 22)
print(o.imie, o.wiek)

Marek 22


### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [28]:
import itertools

def prime_generator():
    n = 2
    while True:
        if all(n % i != 0 for i in range(2, int(n**0.5) + 1)):
            yield n
        n += 1

primes_ending_7 = (p for p in prime_generator() if str(p).endswith('7'))

for p in itertools.islice(primes_ending_7, 10):
    print(p)

7
17
37
47
67
97
107
127
137
157
